In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [4]:
# Import all functions
from pyspark.sql.functions import *

In [5]:
data = [('Alicia', 'Joseph', ['Java', 'Scala', 'Spark'], {'hair': 'black', 'eye': 'brown'}),
        ('Robert', 'Gee', ['Spark', 'Java'], {'hair': 'brown', 'eye': None}),
        ('Mike', 'Bianca', ['CSharp', ''], {'hair': 'red', 'eye': ''}),
        ('John', 'Kumar', None, None),
        ('Jeff', 'L', ['1', '2'], {})]

schema = ['FirstName', 'LastName', 'Languages', 'Properties']

emp1 = spark.createDataFrame(data, schema)
emp1.show(truncate=False)
emp1.printSchema()

+---------+--------+--------------------+-----------------------------+
|FirstName|LastName|Languages           |Properties                   |
+---------+--------+--------------------+-----------------------------+
|Alicia   |Joseph  |[Java, Scala, Spark]|{eye -> brown, hair -> black}|
|Robert   |Gee     |[Spark, Java]       |{eye -> NULL, hair -> brown} |
|Mike     |Bianca  |[CSharp, ]          |{eye -> , hair -> red}       |
|John     |Kumar   |NULL                |NULL                         |
|Jeff     |L       |[1, 2]              |{}                           |
+---------+--------+--------------------+-----------------------------+

root
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



### `explode(col)`
- When an array is passed, it creates a new column with each element of the array in a separate row
- When a map is passed, it creates creates 2 new columns, 1 for key and 1 for value
- It ignores the NULL elements

In [8]:
# For array
emp1.select('FirstName', explode('Languages')).show()

+---------+------+
|FirstName|   col|
+---------+------+
|   Alicia|  Java|
|   Alicia| Scala|
|   Alicia| Spark|
|   Robert| Spark|
|   Robert|  Java|
|     Mike|CSharp|
|     Mike|      |
|     Jeff|     1|
|     Jeff|     2|
+---------+------+



In [9]:
# For map
emp1.select('FirstName', explode('Properties')).show()

+---------+----+-----+
|FirstName| key|value|
+---------+----+-----+
|   Alicia| eye|brown|
|   Alicia|hair|black|
|   Robert| eye| NULL|
|   Robert|hair|brown|
|     Mike| eye|     |
|     Mike|hair|  red|
+---------+----+-----+



### `explode_outer(col)`
- Same as explode() and the only difference is it retains NULL elements

In [11]:
# For array
emp1.select('FirstName', explode_outer('Languages')).show()

+---------+------+
|FirstName|   col|
+---------+------+
|   Alicia|  Java|
|   Alicia| Scala|
|   Alicia| Spark|
|   Robert| Spark|
|   Robert|  Java|
|     Mike|CSharp|
|     Mike|      |
|     John|  NULL|
|     Jeff|     1|
|     Jeff|     2|
+---------+------+



In [10]:
# For map
emp1.select('FirstName', explode_outer('Properties')).show()

+---------+----+-----+
|FirstName| key|value|
+---------+----+-----+
|   Alicia| eye|brown|
|   Alicia|hair|black|
|   Robert| eye| NULL|
|   Robert|hair|brown|
|     Mike| eye|     |
|     Mike|hair|  red|
|     John|NULL| NULL|
|     Jeff|NULL| NULL|
+---------+----+-----+



### `posexplode(col)`
- Same as explode() and the only difference is it creates additional column named pos which has indexes of each element

In [13]:
# For array
emp1.select('FirstName', posexplode('Languages')).show()

+---------+---+------+
|FirstName|pos|   col|
+---------+---+------+
|   Alicia|  0|  Java|
|   Alicia|  1| Scala|
|   Alicia|  2| Spark|
|   Robert|  0| Spark|
|   Robert|  1|  Java|
|     Mike|  0|CSharp|
|     Mike|  1|      |
|     Jeff|  0|     1|
|     Jeff|  1|     2|
+---------+---+------+



In [14]:
# For map
emp1.select('FirstName', posexplode('Properties')).show()

+---------+---+----+-----+
|FirstName|pos| key|value|
+---------+---+----+-----+
|   Alicia|  0| eye|brown|
|   Alicia|  1|hair|black|
|   Robert|  0| eye| NULL|
|   Robert|  1|hair|brown|
|     Mike|  0| eye|     |
|     Mike|  1|hair|  red|
+---------+---+----+-----+



### `flatten(col)`
- Creates a single array from an array of arrays.
- If a structure of nested arrays is deeper than two levels, only one level of nesting is removed.

In [31]:
# Flattening fimple array of array
df = spark.createDataFrame([([[1, 2, 3], [4, 5], [6]],)], ['data'])
df.show(truncate=False)
df.select(flatten('data')).show(truncate=False)

+------------------------+
|data                    |
+------------------------+
|[[1, 2, 3], [4, 5], [6]]|
+------------------------+

+------------------+
|flatten(data)     |
+------------------+
|[1, 2, 3, 4, 5, 6]|
+------------------+



In [32]:
# Flattening an array with more than two levels of nesting
df = spark.createDataFrame([([[[1, 2], [3, 4]], [[5, 6], [7, 8]]],)], ['data'])
df.show(truncate=False)
df.select(flatten(df.data)).show(truncate=False)

+------------------------------------+
|data                                |
+------------------------------------+
|[[[1, 2], [3, 4]], [[5, 6], [7, 8]]]|
+------------------------------------+

+--------------------------------+
|flatten(data)                   |
+--------------------------------+
|[[1, 2], [3, 4], [5, 6], [7, 8]]|
+--------------------------------+

